# Session dynamics: single neurons and latent population activity

Second pass on the Neuropixel open-field session, without behavior. Two questions:

1. **Single neurons.** What does each good unit do across the 95 minutes: stable, drifting, slowly fluctuating, bursty? Do units fall into groups that co-fluctuate?
2. **Population.** What are the dominant latent dimensions of activity, how many are there, and does the population move through discrete states? What are the coordinated dips seen in the first notebook?

Everything here is descriptive and will become the "behavior-free" baseline that the video, once synced, has to explain.

In [ ]:
%matplotlib inline
import sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.signal import welch
from sklearn.decomposition import PCA
from hmmlearn.hmm import GaussianHMM
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd()))
import npx_io as io

DATA_DIR = Path('../../preliminary data /Neuropixel_open_field').resolve()
sns.set_context('notebook'); sns.set_style('ticks')
C_GOOD, C_NARROW, C_BROAD = '#2a9d8f', '#e76f51', '#457b9d'

ci = io.load_cluster_info(DATA_DIR)
st, sc = io.load_spikes(DATA_DIR)
T_END = st.max()
good = ci[ci.label == 'good'].sort_values('depth_um')
units = good.index.to_numpy()
depth = good.depth_um.to_numpy()
narrow = (good.duration_ms < 0.45).to_numpy()
print(f'{len(units)} good units, {narrow.sum()} narrow-spiking, {T_END/60:.1f} min')

## Binned rates and artifact check
1 s bins for slow session structure, 20 ms bins for intrinsic timescales. Before anything else, scan the population rate for artifact bins (many units firing implausibly fast in the same second) and trim the analysis window before the first one.

In [ ]:
BIN = 1.0
counts_all, edges_all = io.bin_spikes(st, sc, units, BIN, t_max=T_END)
pop_all = counts_all.mean(0) / BIN
med, mad = np.median(pop_all), 1.4826 * np.median(np.abs(pop_all - np.median(pop_all)))
art = np.where(pop_all > med + 6 * mad)[0]
print(f'population rate median {med:.2f} Hz, robust SD {mad:.2f} Hz')
if len(art):
    print(f'{len(art)} artifact bin(s) with rate > median + 6 robust SD at t = {edges_all[art]} s '
          f'(peak {pop_all[art].max():.0f} Hz, {(counts_all[:, art].max(1) > 300).sum()} units > 300 spikes/s)')
else:
    print('no artifact bins (population rate never exceeds median + 6 robust SD)')

# analysis window: whole minutes, stopping 10 s before the first artifact and dropping the final minute
T_USE = np.floor(min(T_END - 60, edges_all[art].min() - 10 if len(art) else np.inf) / 60) * 60
counts, edges = io.bin_spikes(st, sc, units, BIN, t_max=T_USE)
rates = counts / BIN                          # (units, seconds)
t = edges[:-1] + BIN / 2
tmin = t / 60
pop = rates.mean(0)
print(f'analysis window 0 to {T_USE:.0f} s ({T_USE/60:.0f} min); rates {rates.shape}')
fig, ax = plt.subplots(figsize=(15, 3))
ax.plot(edges_all[:-1] / 60, pop_all, lw=.6, color='k'); ax.axvline(T_USE / 60, color='r', lw=1, label='analysis window end')
ax.set_yscale('log'); ax.set_xlabel('session time (min)'); ax.set_ylabel('pop rate (Hz, 1 s bins)'); ax.legend(frameon=False)
sns.despine(); plt.tight_layout()

## 1. Single neurons across the session

### 1a. Example time courses
Sixteen units spanning the depth range, 30 s smoothing. Look for slow drifts, step changes, and the shared dips.

In [ ]:
sm30 = io.smooth_rows(rates, 30)
pick = units[np.linspace(0, len(units) - 1, 16).astype(int)]
fig, axes = plt.subplots(8, 2, figsize=(15, 14), sharex=True)
for ax, u in zip(axes.T.ravel(), pick):
    i = np.where(units == u)[0][0]
    ax.plot(tmin, sm30[i], lw=.8, color=C_NARROW if narrow[i] else C_BROAD)
    ax.set_title(f'unit {u}  {depth[i]:.0f} µm  {"narrow" if narrow[i] else "broad"}', fontsize=9, loc='left')
    ax.set_ylim(0, None)
for ax in axes[-1]: ax.set_xlabel('session time (min)')
sns.despine(); plt.tight_layout()

### 1b. Session-level statistics per unit

| statistic | what it captures |
|---|---|
| `slope_per_hr` | linear trend in rate, as a fraction of the mean rate per hour (drift or slow state change) |
| `cv_1min` | coefficient of variation across 1-min bins (how variable the unit is on slow timescales) |
| `slow_frac` | fraction of 1-s rate variance that survives 60-s smoothing (slow vs fast fluctuation) |
| `pop_corr` | correlation with the population mean rate (how much the unit follows the global state) |
| `burst_frac` | fraction of ISIs below 10 ms |
| `cv2` | local ISI irregularity (1 = Poisson-like, less than 1 = regular) |
| `tau_s` | intrinsic timescale: exponential decay constant fitted to the 50-ms spike-count autocorrelation over lags 50 ms to 1 s, zero lag excluded (Murray et al. 2014) |

In [ ]:
def unit_stats():
    hrs = t / 3600
    X = np.c_[np.ones_like(hrs), hrs]
    beta = np.linalg.lstsq(X, rates.T, rcond=None)[0]            # (2, units)
    mean = rates.mean(1)
    r1m = rates.reshape(len(units), -1, 60).mean(2)               # 1-min rates
    sm60 = io.smooth_rows(rates, 60)
    pop_z = (pop - pop.mean()) / pop.std()
    Z = io.zscore_rows(rates)
    spk = io.spikes_by_unit(st, sc, units)
    burst, cv2 = [], []
    for u in units:
        isi = np.diff(spk[u])
        burst.append((isi < 0.01).mean())
        cv2.append(np.mean(2 * np.abs(np.diff(isi)) / (isi[1:] + isi[:-1])))
    c50, _ = io.bin_spikes(st, sc, units, 0.05, t_max=T_USE)
    tau = io.rate_autocorr_timescale(c50, 0.05, max_lag_s=1.0)
    return pd.DataFrame(dict(
        depth_um=depth, narrow=narrow, mean_hz=mean,
        slope_per_hr=beta[1] / (mean + 1e-6),
        cv_1min=r1m.std(1) / (r1m.mean(1) + 1e-6),
        slow_frac=sm60.var(1) / (rates.var(1) + 1e-9),
        pop_corr=(Z * pop_z).mean(1),
        burst_frac=burst, cv2=cv2, tau_s=tau), index=units)

t0 = time.time(); stats = unit_stats(); print(f'{time.time()-t0:.1f} s')
stats.describe().T[['mean', '25%', '50%', '75%']].round(3)

In [ ]:
cols = ['slope_per_hr', 'cv_1min', 'slow_frac', 'pop_corr', 'burst_frac', 'cv2', 'tau_s']
fig, axes = plt.subplots(2, len(cols), figsize=(3.2 * len(cols), 7))
for j, c in enumerate(cols):
    v = stats[c]
    bins = 40 if c != 'tau_s' else np.logspace(np.log10(0.025), np.log10(5), 30)
    axes[0, j].hist(v[~stats.narrow], bins=bins, color=C_BROAD, alpha=.8, label='broad')
    axes[0, j].hist(v[stats.narrow], bins=bins, color=C_NARROW, alpha=.8, label='narrow')
    axes[0, j].set_title(c, fontsize=10)
    axes[1, j].scatter(v, stats.depth_um, s=8, c=np.where(stats.narrow, C_NARROW, C_BROAD), alpha=.7)
    axes[1, j].set_xlabel(c)
    if c == 'tau_s': axes[0, j].set_xscale('log'); axes[1, j].set_xscale('log')
axes[0, 0].legend(frameon=False, fontsize=8); axes[0, 0].set_ylabel('n units'); axes[1, 0].set_ylabel('depth (µm)')
print(f'{(stats.slope_per_hr.abs() > 0.5).sum()} units change by >50% of their mean rate per hour')
print(f'median |pop_corr| = {stats.pop_corr.abs().median():.2f};  '
      f'{(stats.pop_corr < 0).sum()} units are anti-correlated with the population')
sns.despine(); plt.tight_layout()
print(f'tau fitted for {stats.tau_s.notna().sum()} units; median narrow {stats.tau_s[stats.narrow].median()*1000:.0f} ms, broad {stats.tau_s[~stats.narrow].median()*1000:.0f} ms')

### 1c. Groups of co-fluctuating units
Correlate the 30 s smoothed, z-scored rate profiles and cluster hierarchically. The interesting outcome is whether there is one global mode (everything follows the population) or several distinct profiles, for example one group that drifts down while another drifts up.

In [ ]:
Z30 = io.zscore_rows(sm30)
R = np.corrcoef(Z30)
link = linkage(Z30, method='average', metric='correlation')
N_CLUST = 5
lab = fcluster(link, N_CLUST, criterion='maxclust')
order_c = dendrogram(link, no_plot=True)['leaves']

fig = plt.figure(figsize=(16, 6))
gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.6, .6])
ax = fig.add_subplot(gs[0])
im = ax.imshow(R[np.ix_(order_c, order_c)], cmap='RdBu_r', vmin=-.6, vmax=.6)
ax.set_title('unit x unit correlation of session profile (clustered order)'); ax.set_xlabel('unit'); ax.set_ylabel('unit')
plt.colorbar(im, ax=ax, fraction=.046)
ax = fig.add_subplot(gs[1])
pal = sns.color_palette('tab10', N_CLUST)
for k in range(1, N_CLUST + 1):
    m = lab == k
    ax.plot(tmin, Z30[m].mean(0) + 3 * (k - 1), color=pal[k - 1], lw=1, label=f'cluster {k} (n={m.sum()})')
ax.set_xlabel('session time (min)'); ax.set_ylabel('mean z (offset per cluster)'); ax.legend(frameon=False, fontsize=8, loc='upper right')
ax.set_title('mean session profile per cluster')
ax = fig.add_subplot(gs[2])
for k in range(1, N_CLUST + 1):
    m = lab == k
    ax.scatter(np.random.uniform(k - .3, k + .3, m.sum()), depth[m], s=8, color=pal[k - 1])
ax.set_xlabel('cluster'); ax.set_ylabel('depth (µm)'); ax.set_title('depth by cluster')
sns.despine(); plt.tight_layout()
pd.crosstab(pd.Series(lab, name='cluster'), pd.Series(np.where(narrow, 'narrow', 'broad'), name='type'))

## 2. Population dynamics

### 2a. Pairwise correlations at 1 s
Sorted by depth, so spatial structure along the probe shows up as blocks. The right panel asks whether correlation falls off with distance along the probe.

In [ ]:
Z1 = io.zscore_rows(io.smooth_rows(rates, 5))
R1 = np.corrcoef(Z1)
iu = np.triu_indices(len(units), 1)
dd = np.abs(depth[:, None] - depth[None, :])[iu]; rr = R1[iu]
bins = np.arange(0, 3400, 200)
prof = [rr[(dd >= a) & (dd < b)].mean() for a, b in zip(bins[:-1], bins[1:])]
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw=dict(width_ratios=[1.3, 1]))
im = axes[0].imshow(R1, cmap='RdBu_r', vmin=-.4, vmax=.4, extent=[depth[0], depth[-1], depth[-1], depth[0]])
axes[0].set_xlabel('depth (µm)'); axes[0].set_ylabel('depth (µm)'); axes[0].set_title('pairwise correlation, 5 s smoothed, sorted by depth')
plt.colorbar(im, ax=axes[0], fraction=.046)
axes[1].plot(bins[:-1] + 100, prof, 'o-', color='k'); axes[1].axhline(0, color='k', lw=.5)
axes[1].set_xlabel('distance along probe (µm)'); axes[1].set_ylabel('mean pairwise correlation'); axes[1].set_title('correlation vs distance')
print(f'mean pairwise correlation {rr.mean():.3f}; {(rr > 0.3).mean()*100:.1f}% of pairs above 0.3')
sns.despine(); plt.tight_layout()

### 2b. PCA on the z-scored population
How many dimensions carry the slow activity? The participation ratio gives an effective dimensionality. If PC1 is nearly identical to the population mean rate, the dominant mode is a global gain signal, which is common in cortex and usually tracks arousal or movement.

In [ ]:
pca = PCA(n_components=30).fit(Z1.T)
S = pca.transform(Z1.T)                   # (time, PCs)
ev = pca.explained_variance_ratio_
lam = pca.explained_variance_
pr = lam.sum() ** 2 / (lam ** 2).sum()
r_pc1_pop = np.corrcoef(S[:, 0], io.smooth_rows(pop[None], 5)[0])[0, 1]
print(f'variance explained: PC1 {ev[0]*100:.1f}%, top 5 {ev[:5].sum()*100:.1f}%, top 10 {ev[:10].sum()*100:.1f}%')
print(f'participation ratio (30 PCs) = {pr:.1f};  corr(PC1, population rate) = {r_pc1_pop:.2f}')

fig, axes = plt.subplots(2, 2, figsize=(15, 8), gridspec_kw=dict(width_ratios=[1, 2.2]))
axes[0, 0].plot(np.arange(1, 31), ev * 100, 'o-', color='k'); axes[0, 0].set_xlabel('PC'); axes[0, 0].set_ylabel('% variance')
axes[0, 0].set_title('scree')
for k in range(5):
    axes[0, 1].plot(tmin, S[:, k] / S[:, k].std() - 3.5 * k, lw=.7, label=f'PC{k+1}')
axes[0, 1].set_yticks([]); axes[0, 1].set_xlabel('session time (min)'); axes[0, 1].legend(frameon=False, fontsize=8, ncol=5, loc='upper right')
axes[0, 1].set_title('top 5 PC scores over the session')
W = pca.components_[:5].T                 # (units, PCs) loadings
for k in range(3):
    axes[1, 0].scatter(W[:, k], depth, s=6, alpha=.6, label=f'PC{k+1}')
axes[1, 0].axvline(0, color='k', lw=.5); axes[1, 0].set_xlabel('loading'); axes[1, 0].set_ylabel('depth (µm)'); axes[1, 0].legend(frameon=False, fontsize=8)
axes[1, 0].set_title('loadings vs depth')
sc_ = axes[1, 1].scatter(S[:, 0], S[:, 1], c=tmin, cmap='viridis', s=3)
axes[1, 1].set_xlabel('PC1'); axes[1, 1].set_ylabel('PC2'); axes[1, 1].set_title('trajectory, colored by session time')
plt.colorbar(sc_, ax=axes[1, 1], label='min')
sns.despine(); plt.tight_layout()
print('mean loading on PC1: narrow', W[narrow, 0].mean().round(3), ' broad', W[~narrow, 0].mean().round(3))

### 2c. Timescales of the latent activity
Power spectrum of the population rate and PC1 from 100 ms bins. Peaks or a knee tell us what timescale dominates (seconds = movement bouts, tens of seconds = arousal or behavioral states).

In [ ]:
c100, e100 = io.bin_spikes(st, sc, units, 0.1, t_max=T_USE)
pop100 = c100.mean(0) / 0.1
pc1_100 = io.zscore_rows(c100 / 0.1) .T @ pca.components_[0]
fig, ax = plt.subplots(figsize=(8, 4.5))
for sig, name in [(pop100, 'population rate'), (pc1_100, 'PC1')]:
    f, p = welch(sig - sig.mean(), fs=10, nperseg=2 ** 12)
    ax.loglog(f[1:], p[1:] / p[1:].max(), lw=1, label=name)
for per, lbl in [(1, '1 s'), (10, '10 s'), (60, '1 min'), (600, '10 min')]:
    ax.axvline(1 / per, color='k', lw=.4, ls=':'); ax.text(1 / per, 1.2, lbl, ha='center', fontsize=8)
ax.set_xlabel('frequency (Hz)'); ax.set_ylabel('normalized power'); ax.legend(frameon=False)
sns.despine(); plt.tight_layout()

### 2d. Discrete states: Gaussian HMM on the top PCs
Fit HMMs with 2 to 6 states on the top 10 PC scores (1 s bins, 5 s smoothed) and pick by BIC. Then look at when each state occurs, how long it lasts, and what each unit does in each state. Treat the state count as descriptive; the BIC minimum is often generous for smoothed data.

Note after the first run: BIC keeps falling up to 6 states, the states are ordered by population rate in small steps, and dwell times cluster around 10 s. That pattern means the HMM is discretizing a continuous arousal-like variable rather than finding a few distinct modes. Treat the state sequence as a convenient segmentation, not as evidence for discrete states.

In [ ]:
X = S[:, :10]
fits = {}
for n in range(2, 7):
    m = GaussianHMM(n_components=n, covariance_type='diag', n_iter=300, random_state=0).fit(X)
    fits[n] = (m, m.bic(X))
    print(f'{n} states: BIC {fits[n][1]:.0f}')
N_STATES = min(fits, key=lambda n: fits[n][1])
hmm = fits[N_STATES][0]
states = hmm.predict(X)
# relabel states by mean population rate so state 0 = lowest firing
order_s = np.argsort([pop[states == s].mean() for s in range(N_STATES)])
remap = np.zeros(N_STATES, int); remap[order_s] = np.arange(N_STATES)
states = remap[states]
print(f'chosen: {N_STATES} states')

In [ ]:
pal = sns.color_palette('Set2', N_STATES)
fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True, gridspec_kw=dict(height_ratios=[.5, 1, 3]))
axes[0].imshow(states[None], aspect='auto', cmap=plt.matplotlib.colors.ListedColormap(pal), extent=[0, tmin[-1], 0, 1], vmin=-.5, vmax=N_STATES-.5)
axes[0].set_yticks([]); axes[0].set_title('HMM state sequence')
axes[1].plot(tmin, io.smooth_rows(pop[None], 10)[0], color='k', lw=.8); axes[1].set_ylabel('pop rate (Hz)')
im = axes[2].imshow(io.zscore_rows(io.smooth_rows(rates, 10)), aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2, extent=[0, tmin[-1], len(units), 0])
axes[2].set_ylabel('units (by depth)'); axes[2].set_xlabel('session time (min)')
plt.tight_layout()

occ = np.bincount(states, minlength=N_STATES) / len(states)
dwell = {s: [e - b for b, e in io.find_epochs(states == s, t)] for s in range(N_STATES)}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].bar(range(N_STATES), occ * 100, color=pal); axes[0].set_xlabel('state'); axes[0].set_ylabel('% of session'); axes[0].set_title('occupancy')
for s in range(N_STATES):
    axes[1].hist(dwell[s], bins=np.logspace(0, 3, 30), histtype='step', lw=1.5, color=pal[s], label=f'state {s} (n={len(dwell[s])})')
axes[1].set_xscale('log'); axes[1].set_xlabel('dwell time (s)'); axes[1].set_ylabel('n epochs'); axes[1].legend(frameon=False, fontsize=8); axes[1].set_title('dwell times')
state_rate = np.array([rates[:, states == s].mean(1) for s in range(N_STATES)]).T   # units x states
rel = state_rate / (rates.mean(1, keepdims=True) + 1e-6)
for s in range(N_STATES):
    axes[2].scatter(np.log2(rel[:, s] + 1e-3), depth, s=6, color=pal[s], alpha=.6, label=f'state {s}')
axes[2].axvline(0, color='k', lw=.5); axes[2].set_xlabel('log2(rate in state / session mean)'); axes[2].set_ylabel('depth (µm)'); axes[2].legend(frameon=False, fontsize=8); axes[2].set_title('per-unit rate change by state')
sns.despine(); plt.tight_layout()
print('mean population rate per state (Hz):', np.round([pop[states == s].mean() for s in range(N_STATES)], 2))

### 2e. The coordinated dips
Define dips directly from the population rate (10 s smoothed, below median minus 2.5 robust SD) rather than from the HMM, then average every unit's z-scored rate around dip onset. This asks whether dips are a uniform suppression, or whether some units switch on while the rest switch off, which would look more like a state change (for example rest or grooming) than an artifact.

In [ ]:
pop10 = io.smooth_rows(pop[None], 10)[0]
med10, mad10 = np.median(pop10), 1.4826 * np.median(np.abs(pop10 - np.median(pop10)))
thr = med10 - 2.5 * mad10
dips = io.find_epochs(pop10 < thr, t, min_len_s=5)
print(f'{len(dips)} dips of >= 5 s; durations (s):', np.round([e - b for b, e in dips], 0))
Z10 = io.zscore_rows(io.smooth_rows(rates, 5))
W_PRE, W_POST = 60, 120
segs = []
for b, e in dips:
    i = int(b)
    if i - W_PRE >= 0 and i + W_POST < Z10.shape[1]:
        segs.append(Z10[:, i - W_PRE:i + W_POST])
segs = np.stack(segs)                       # (dips, units, time)
avg = segs.mean(0)
lag = np.arange(-W_PRE, W_POST)
resp = avg[:, W_PRE:W_PRE + 30].mean(1) - avg[:, :W_PRE - 10].mean(1)
order_r = np.argsort(resp)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), gridspec_kw=dict(width_ratios=[1.6, 1, 1]))
im = axes[0].imshow(avg[order_r], aspect='auto', cmap='RdBu_r', vmin=-1.5, vmax=1.5, extent=[lag[0], lag[-1], len(units), 0])
axes[0].axvline(0, color='k', lw=.8); axes[0].set_xlabel('time from dip onset (s)'); axes[0].set_ylabel('units (sorted by response)'); axes[0].set_title(f'mean z-rate around {len(segs)} dip onsets')
plt.colorbar(im, ax=axes[0], fraction=.046)
axes[1].hist(resp[~narrow], bins=40, color=C_BROAD, alpha=.8, label='broad'); axes[1].hist(resp[narrow], bins=40, color=C_NARROW, alpha=.8, label='narrow')
axes[1].axvline(0, color='k', lw=.5); axes[1].set_xlabel('z change (first 30 s of dip minus baseline)'); axes[1].set_ylabel('n units'); axes[1].legend(frameon=False)
axes[2].scatter(resp, depth, s=8, c=np.where(narrow, C_NARROW, C_BROAD), alpha=.7); axes[2].axvline(0, color='k', lw=.5)
axes[2].set_xlabel('z change'); axes[2].set_ylabel('depth (µm)')
print(f'{(resp < -0.5).sum()} units suppressed by >0.5 z, {(resp > 0.5).sum()} units enhanced by >0.5 z')
sns.despine(); plt.tight_layout()

### 2f. Are the dips immobility?
Video motion energy on the ephys clock (see the alignment section of the inspection notebook), averaged around the same dip onsets.

In [ ]:
CACHE = Path('cache')
warp = io.load_video_warp(CACHE)
tm, mot = io.load_motion_energy(CACHE, warp)
mot_1s = io.bin_signal(tm, mot, edges)                      # same 1 s bins as `rates`
mot_z = (mot_1s - np.nanmean(mot_1s)) / np.nanstd(mot_1s)
r_pop = np.corrcoef(np.nan_to_num(mot_z), (pop - pop.mean()) / pop.std())[0, 1]
segs_m = np.stack([mot_z[int(b) - W_PRE:int(b) + W_POST] for b, e in dips if int(b) - W_PRE >= 0 and int(b) + W_POST < len(mot_z)])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(lag, segs_m.T, color='grey', lw=.5, alpha=.6); axes[0].plot(lag, np.nanmean(segs_m, 0), color='k', lw=2, label='mean')
axes[0].axvline(0, color='r', lw=.8); axes[0].set_xlabel('time from dip onset (s)'); axes[0].set_ylabel('motion energy (z)'); axes[0].legend(frameon=False)
axes[0].set_title('video motion around population dips')
axes[1].scatter(mot_z, pop, s=2, alpha=.2, color='k'); axes[1].set_xlabel('motion energy (z, 1 s)'); axes[1].set_ylabel('population rate (Hz)')
axes[1].set_title(f'1 s bins, r = {r_pop:.2f}')
sns.despine(); plt.tight_layout()
print(f'motion during first 30 s of dips vs 60 s before: {np.nanmean(segs_m[:, W_PRE:W_PRE+30]):+.2f} z vs {np.nanmean(segs_m[:, :W_PRE]):+.2f} z')

## 3. What to take from this (first pass, 2026-09-09)

**Single neurons**
* Rates are far from stationary. About 90 of 433 good units change by more than half their mean rate per hour, in both directions, and the examples show ramps, decays, and step changes. Event-locked analyses will need slow drift regressors or session-time splits.
* Slow fluctuations (over 1 min) carry only about 10% of the 1-s rate variance for a typical unit. Most variance is on the seconds timescale, which is where behavior lives.
* Narrow-spiking units follow the population mean far more tightly than broad units (population correlation around 0.4 versus 0.1), and are almost all suppressed during dips. That is the classic signature of fast-spiking interneurons tracking a global arousal or movement signal.

**Population**
* Activity is high dimensional: PC1 carries only 9% of variance, the participation ratio is about 14, and PC1 is only moderately related to the population mean (r about 0.6). There is no single dominant gain mode.
* The unit-by-unit correlation matrix has clear block structure with anticorrelated groups, and hierarchical clustering separates a large cluster that follows the population from a cluster of about 90 units that moves against it.
* The coordinated dips are state switches, not artifacts. Roughly 130 units drop by more than 0.5 z while about 50 units turn on, and the "on" units are spread across depth. Dips last 5 to 60 s and recur every few minutes. Section 2f checks this against video motion energy on the corrected clock: see the printed numbers there for whether dips are immobility.
* The PC1 to PC2 trajectory shows a distinct excursion around minutes 65 to 75 and a drift late in the session, so the network state at the end differs from the start.

**What this means for the approach analysis**
* Drift and the dip state are both large enough to masquerade as stimulus effects if approaches cluster in time. Include session time and a dip or state indicator as nuisance regressors, and split cross-validation folds by time blocks.
* The "dip-on" population is a candidate for a rest-related ensemble. Worth tracking whether those units are also the ones that are quiet during stimulus investigation.
* Motion energy alone correlates with the population rate at about r = 0.5 in 1 s bins. Once tracking exists, replace it with speed and distance-to-wall and regress against the top 10 PCs to see how much of this structure is movement.